# X-ray Galaxy Cluster Mass Inference — Walkthrough

## Abstract

This notebook walks through the full pipeline for inferring **galaxy cluster masses** from simulated **eROSITA** X-ray photon-count images using a CNN regression model with a calibrated uncertainty head. We:

1. Load the multi-energy-band image cube (`(N, 50, 50, 10)`) and the matching cluster catalogue.
2. Inspect the data — mass / redshift distributions, what a single cluster looks like across the 10 energy bands.
3. Standardise inputs and split into train / validation / test sets.
4. Build a configurable CNN that outputs `(μ, σ²)` per cluster.
5. Train it with a heteroscedastic Gaussian negative-log-likelihood loss so the network *learns its own uncertainty*.
6. Evaluate predictions on the test set, including a coverage check on the predicted ±1σ intervals.

The walkthrough is self-contained: it imports project code (`data_loader`, `preprocessing`, `models`, `training`) but reads the catalogue directly from FITS so you don't have to do the FITS→feather conversion needed for the CLI pipeline.

**Audience.** Written for someone who knows ML / TensorFlow but not astrophysics — physical concepts (clusters, M500c, log-mass, photon counts) are explained inline.

In [ ]:
# Cell 2 — data availability check.
# We never auto-download: the files are ~2 GB total. If they are missing, print clear instructions and stop.
import os, sys
from pathlib import Path

# Locate the project root robustly: walk up from CWD looking for a sibling 'physics_data/'
# (works whether the kernel CWD is the project root or 'notebooks/').
_here = Path.cwd()
PROJECT_ROOT = None
for _candidate in (_here, *_here.parents):
    if (_candidate / "physics_data").exists():
        PROJECT_ROOT = _candidate
        break
    if _candidate.name == "xray-cluster-masses":
        PROJECT_ROOT = _candidate
        break
if PROJECT_ROOT is None:
    PROJECT_ROOT = _here.parent if _here.name == "notebooks" else _here

DATA_DIR = str((PROJECT_ROOT / "physics_data").resolve())
IMAGE_FILE = os.path.join(DATA_DIR, "eFEDS_01to18-3dImgs-7946clus-300pix-50pix-ext_det_thr.pickle")
CATALOG_FITS = os.path.join(DATA_DIR, "eFEDS_mock_clusters_catalog_01to18-ext_det_thr.fits")
CATALOG_FEATHER = os.path.join(DATA_DIR, "eFEDS_mock_clusters_catalog_01to18-ext_det_thr.f")

missing = [p for p in (IMAGE_FILE,) if not os.path.exists(p)]
if not (os.path.exists(CATALOG_FITS) or os.path.exists(CATALOG_FEATHER)):
    missing.append(CATALOG_FITS)

if missing:
    print("Required data file(s) not found:")
    for p in missing:
        print(f"  - {p}")
    print()
    print("Create the data directory and download both files into it:")
    print(f"  mkdir -p {DATA_DIR}")
    print(f"  cd {DATA_DIR}")
    print("  wget https://cloud.physik.lmu.de/index.php/s/ParyrwygTYdFcWY/download/eFEDS_01to18-3dImgs-7946clus-300pix-50pix-ext_det_thr.pickle")
    print("  wget https://cloud.physik.lmu.de/index.php/s/5xex5aXqRYSdSMw/download/eFEDS_mock_clusters_catalog_01to18-ext_det_thr.fits")
    raise SystemExit("Stopping: please download the data files and rerun this cell.")

print("Data files found:")
print(f"  images   : {IMAGE_FILE}")
print(f"  catalogue: {CATALOG_FITS if os.path.exists(CATALOG_FITS) else CATALOG_FEATHER}")


In [ ]:
# Cell 3 — imports, organised by category.
import sys, pickle
from pathlib import Path

# numerics & data
import numpy as np
import pandas as pd
from astropy.table import Table
from sklearn.model_selection import train_test_split

# plotting
import matplotlib.pyplot as plt

# deep learning
import tensorflow as tf
from tensorflow import keras
from keras import layers, Input, Model, Sequential
from keras.layers import RandomFlip, RandomRotation, RandomZoom
from keras.callbacks import EarlyStopping

# project code (one level up)
sys.path.insert(0, str(Path.cwd().parent))
from preprocessing import Preprocessor      # noqa: E402
from models import ModelFactory             # noqa: E402
from training import Trainer                # noqa: E402

print("TensorFlow:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices("GPU"))

## 1. Load the data

Two pieces:

- **Image cube** — a Python pickle of a dict whose key `gsm_3dImgs` holds a list of 7,946 cluster cutouts. Each cutout is a `(50, 50, 10)` `float32` array: 50×50 pixels in 10 X-ray energy bands. Higher pixel values mean more X-ray photons detected at that sky position in that band.
- **Catalogue** — a FITS table with one row per cluster. We need the mass `M500c` (solar masses inside the radius enclosing 500× the critical density) and the redshift `z`, plus optional auxiliaries (gas temperature `kT`, soft-band flux/luminosity, …). We work with `log10(M500c / M_sun)` because masses span several decades.

We read the FITS file directly via `astropy.table.Table` so this notebook works with the file as downloaded — no FITS→feather conversion required.

In [ ]:
with open(IMAGE_FILE, "rb") as f:
    image_data = pickle.load(f)

images = np.asarray(image_data["gsm_3dImgs"], dtype=np.float32)
print("Image cube shape:", images.shape, " dtype:", images.dtype)

if os.path.exists(CATALOG_FITS):
    catalog_df = Table.read(CATALOG_FITS).to_pandas()
else:
    catalog_df = pd.read_feather(CATALOG_FEATHER)

print("Catalogue rows:", len(catalog_df), " columns:", len(catalog_df.columns))
catalog_df.head()

## 2. Explore the data

Two sanity checks before we model anything.

- **Mass and redshift histograms** tell us the prior distribution the network is fitting against. Cluster masses span ~`10^13` – `10^15` M_sun (so `log10(M)` ≈ 13–15) and redshifts roughly 0–1.5 in this eFEDS-like simulation. Knowing the support is important: the network will be unreliable outside the densely-sampled region, and we should be careful about boundary effects in evaluation.
- **A single cluster across all 10 energy bands** shows that different bands carry *different* information. Soft-band photons trace cooler gas in the outskirts; hard-band photons trace the hot core. A multi-channel CNN can in principle exploit all of this jointly — that's the whole reason we don't collapse to a single image.

In [ ]:
log_mass = np.log10(catalog_df["M500c"].values)
redshift = catalog_df["z"].values

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(log_mass, bins=40, color="#539ecd")
axes[0].set_xlabel(r"$\log_{10}(M_{500c}/M_\odot)$")
axes[0].set_ylabel("Number of clusters")
axes[0].set_title("Cluster mass distribution")
axes[0].grid(alpha=0.3)

axes[1].hist(redshift, bins=40, color="#ffb877")
axes[1].set_xlabel("Redshift z")
axes[1].set_ylabel("Number of clusters")
axes[1].set_title("Redshift distribution")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualise one cluster across all 10 energy bands.
# We pick a high-mass, low-redshift cluster so it is bright and well-resolved.
bright_idx = int(np.argmax(log_mass / (1.0 + redshift)))
cluster_img = images[bright_idx]      # shape (50, 50, 10)
n_channels = cluster_img.shape[-1]    # 10 energy bands

fig, axes = plt.subplots(2, 5, figsize=(13, 5.5))
for ch_idx, ax in enumerate(axes.flat):
    ax.imshow(cluster_img[..., ch_idx], origin="lower", cmap="inferno")
    ax.set_title(f"Channel {ch_idx}")
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(
    f"Cluster #{bright_idx}: log10(M500c)={log_mass[bright_idx]:.2f}, z={redshift[bright_idx]:.3f}",
    y=1.02,
)
plt.tight_layout()
plt.show()

## 3. Train / validation / test split

We use a **fixed-seed random split** of 70 / 15 / 15. There's no obvious astrophysical reason to stratify (clusters are independent and the simulation has no time/sky correlations we need to preserve), and a fixed seed gives reproducibility across the variants we trained for the README results table.

We use the project's `Preprocessor` here, which:

- Stores all features in a single dict (`mass`, `redshift`, all catalogue columns).
- Normalises images per-image (zero mean, unit variance). Per-image normalisation removes overall brightness — the CNN then has to learn from morphology, not from total photon count.
- Standardises mass (and any auxiliary scalar) on the *full* dataset before the split. (Note: a strictly leakage-free pipeline would fit the standardiser on the train split only; we keep this consistent with the rest of the repo.)
- Returns three new `Preprocessor` instances for train / val / test.

In [ ]:
feature_dict = {
    "mass":     np.log10(catalog_df["M500c"].values),
    "redshift": catalog_df["z"].values,
    "index":    catalog_df.index.values,
}
for col in catalog_df.columns:
    if col not in feature_dict:
        feature_dict[col] = catalog_df[col].values

prep = Preprocessor(images, feature_dict)
prep.normalize_images(mode="per_image")
prep.normalize_feature("mass")
prep.normalize_feature("redshift")

train_set, val_set, test_set = prep.split(test_size=0.15, val_size=0.15, random_state=42)

x_train, y_train = train_set.get_xy("mass")
x_val,   y_val   = val_set.get_xy("mass")
x_test,  y_test  = test_set.get_xy("mass")
print(f"train: {x_train.shape},  val: {x_val.shape},  test: {x_test.shape}")

## 4. CNN architecture

A standard image regression CNN with one twist: the **final layer outputs two numbers per sample** — a predicted mean μ and a predicted variance σ² — instead of a single point estimate. The variance head goes through a sigmoid so it stays positive and bounded.

The architecture below mirrors the best Optuna-tuned configuration on the image-only branch (3 conv layers, 2 dense layers, dropout). For the walkthrough we use the *image-only* version (no auxiliary inputs) so the demo is fast to read; `ModelFactory.build_cnn` also supports concatenating scalar inputs (redshift, kT) into a small head, which is what the best-performing variant uses.

Design choices worth flagging:

- **No max-pooling** in the best config: the inputs are already small (50×50), and pooling discards spatial information that turned out to matter. Optuna picked this autonomously.
- **Batch normalisation only on the last conv layer**, not after every conv. With small images and deep dropout downstream, more BN didn't help.
- **Heavy dropout (0.4)** in the dense layers controls the otherwise huge parameter count after `Flatten` (320,000 → 64).

In [ ]:
conv_layers = [
    {"filters": 32,  "kernel_size": (5, 5), "padding": "same", "activation": "relu", "pool": False, "batchnorm": False},
    {"filters": 32,  "kernel_size": (2, 2), "padding": "same", "activation": "relu", "pool": False, "batchnorm": False},
    {"filters": 128, "kernel_size": (2, 2), "padding": "same", "activation": "relu", "pool": False, "batchnorm": True},
]
fnn_layers = [
    {"units": 64,  "activation": "relu", "dropout": True, "dropout_rate": 0.45},
    {"units": 256, "activation": "relu", "dropout": True, "dropout_rate": 0.40},
]

# Build the image-only model (no auxiliary inputs).
model = ModelFactory.build_cnn(
    image_shape=x_train.shape[1:],
    conv_config_layers=conv_layers,
    fnn_config_layers=fnn_layers,
    output_dir="_walkthrough_run",   # writes model_summary.txt here
)
model.summary()

## 5. Custom loss — Gaussian negative log-likelihood

We don't just want the network to predict the mass — we want it to tell us how confident it is. So instead of a plain MSE loss, we treat the network's output as a Gaussian likelihood and minimise the negative log-likelihood:

$$
\mathcal{L}(y, \mu, \sigma^2) = \frac{1}{2}\log(\sigma^2 + \epsilon) + \frac{(y - \mu)^2}{2(\sigma^2 + \epsilon)}
$$

Two terms with opposing pulls:

- The **MSE-like term** punishes wrong predictions. For a fixed σ², minimising it is equivalent to MSE — but here it's *weighted* by `1/σ²`, so the network can choose to be "less confident" on hard samples to reduce the penalty.
- The **log term** punishes claiming infinite uncertainty. Without it, the network would minimise the loss trivially by predicting σ² → ∞.

At the optimum, σ² should track the actual squared error — i.e. the network learns calibrated, input-dependent uncertainties for free. We monitor MSE on the mean head as a separate metric for interpretability.

In [ ]:
# The implementation lives in training.Trainer.loss_nll. Here it is verbatim, for readability:
def loss_nll(y_true, y_pred):
    """Heteroscedastic Gaussian NLL loss. y_pred[:, 0] = mean, y_pred[:, 1] = variance (positive)."""
    y_true = tf.squeeze(y_true)
    mu  = y_pred[:, 0]
    var = y_pred[:, 1]
    jitter = 1e-8
    return tf.reduce_mean(
        0.5 * tf.math.log(var + jitter) + 0.5 * tf.square(y_true - mu) / (var + jitter)
    )

## 6. Training

Adam optimiser at the Optuna-tuned learning rate (`2.6e-4`), batch size 64, up to 100 epochs, **early stopping** with patience 20 on validation loss (and we restore the best weights). Patience is generous because NLL training tends to oscillate before it converges on a stable variance.

For the walkthrough we set `MAX_EPOCHS` lower so the demo runs quickly. To reproduce the README results, set it back to 100.

In [ ]:
MAX_EPOCHS = 30   # README results used 100

trainer = Trainer(model, output_dir="_walkthrough_run")
trainer.compile(
    optimizer=keras.optimizers.Adam(learning_rate=2.615e-4),
    loss=Trainer.loss_nll,
    metrics=[Trainer.mse_mean],
)
history = trainer.train(
    x_train, y_train,
    x_val,   y_val,
    batch_size=64,
    epochs=MAX_EPOCHS,
    patience=20,
)

In [ ]:
# Plot loss curves (NLL and MSE) by hand so we can see them inline.
epochs = range(1, len(history.history["loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs, history.history["mse_mean"],     label="train")
axes[0].plot(epochs, history.history["val_mse_mean"], label="val", linestyle="--")
axes[0].set(title="MSE on mean head", xlabel="epoch", ylabel="MSE")
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(epochs, history.history["loss"],     label="train")
axes[1].plot(epochs, history.history["val_loss"], label="val", linestyle="--")
axes[1].set(title="Gaussian NLL loss", xlabel="epoch", ylabel="NLL")
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Test-set predictions & uncertainty calibration

Two quality checks on the held-out test set:

- **Predicted vs. true mass** — the standard scatter plot. Points should lie close to the y=x line. We add the predicted ±1σ as error bars and colour points by whether the truth lies inside their interval.
- **Coverage** — the fraction of test points whose true mass falls inside the predicted ±1σ interval. For a *well-calibrated* Gaussian, this should be ≈68.3%. Lower coverage means the network is overconfident; higher means it's hedging.

In [ ]:
y_pred_norm = trainer.predict(x_test)            # (N, 2) in normalised mass units
mu_norm    = y_pred_norm[:, 0]
var_norm   = y_pred_norm[:, 1]
sigma_norm = np.sqrt(var_norm)

# Denormalise back to log10(M500c/M_sun)
mu      = prep.denormalize_feature("mass", mu_norm)
y_true  = prep.denormalize_feature("mass", y_test)
sigma   = sigma_norm * (prep.feature_stats["mass"]["std"] + 1e-8)

coverage = Trainer.check_uncertainty(y_true, mu, sigma)
test_mse = float(np.mean((mu - y_true) ** 2))
print(f"Test MSE on log10(M500c): {test_mse:.4f}")
print(f"Coverage within +/- 1 sigma: {coverage*100:.1f}% (well-calibrated: ~68.3%)")

fig, (ax_main, ax_pull) = plt.subplots(
    2, 1, figsize=(7, 7),
    gridspec_kw={"height_ratios": [4, 1], "hspace": 0.05}, sharex=True,
)

inside  = (y_true >= mu - sigma) & (y_true <= mu + sigma)
outside = ~inside
ax_main.errorbar(y_true[inside],  mu[inside],  yerr=sigma[inside],  fmt="o", color="#539ecd", alpha=0.5, label="within \u00b11\u03c3")
ax_main.errorbar(y_true[outside], mu[outside], yerr=sigma[outside], fmt="o", color="#ffb877", alpha=0.5, label="outside \u00b11\u03c3")
lim = [y_true.min(), y_true.max()]
ax_main.plot(lim, lim, "k--", label="y = x")
ax_main.set_ylabel(r"Predicted $\log_{10}(M_{500c}/M_\odot)$")
ax_main.set_title(f"Predicted vs. true mass — coverage \u00b11\u03c3 = {coverage*100:.1f}%")
ax_main.legend(); ax_main.grid(alpha=0.3)

pull = (mu - y_true) / sigma
ax_pull.scatter(y_true, pull, alpha=0.5, s=10)
ax_pull.axhline(0, color="black", linestyle="--")
ax_pull.set_xlabel(r"True $\log_{10}(M_{500c}/M_\odot)$")
ax_pull.set_ylabel("pull (\u03bc\u2212y)/\u03c3")
ax_pull.set_ylim(-3, 3)
ax_pull.grid(alpha=0.3)
plt.show()

## 8. Conclusions and key findings

- **CNNs work for cluster-mass regression.** Even the image-only baseline produces predictions tightly correlated with the true `log10(M500c)`, with no hand-engineered features and no scaling-relation assumptions.
- **Heteroscedastic NLL gives free, calibrated uncertainties.** Coverage on the held-out set lands close to the 68% target without any post-hoc calibration. The pull plot is the cleanest single diagnostic: it should look like noise around 0 with std ≈ 1, and outliers there are diagnostic of model bias rather than just "hard examples".
- **Auxiliary scalars help, augmentation helps.** Comparing the variants in the README's *Results* section, the best test-set MSE comes from the Optuna-tuned model that combines (image, redshift, kT) inputs with random flips/rotations/zoom on the training set. Redshift alone reduces residual variance noticeably — physically intuitive, since at fixed observed brightness a more distant cluster must be more massive.
- **The architecture surprises** were learnt by Optuna, not designed: no max-pooling, batchnorm only at the last conv layer, large dense layers behind heavy dropout. A fully manual design would have been more conservative, and worse.
- **Caveat: simulated data only.** The training set is mock eROSITA-eFEDS — clean PSF model, known backgrounds, perfect labels. A serious application to real eROSITA data would need a careful study of domain shift (PSF, masking, foreground / background contaminants, mis-centring) before trusting the predicted uncertainties on the sky.